In [ ]:
import numpy as np
import time

# 1. Реализация на циклах
class SlowDetectorAndSource:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5, att_len=8.0, Y=10000):
        self.R, self.N, self.r, self.a, self.att_len, self.Y = R, N, r, a, att_len, Y
        phi = np.pi * (5.0**0.5 - 1.0)
        self.pmts = []
        for i in range(N):
            y = 1 - (i / float(N - 1)) * 2
            r_at_y = np.sqrt(1 - y * y)
            theta = phi * i
            self.pmts.append([np.cos(theta)*r_at_y*R, y*R, np.sin(theta)*r_at_y*R])

    def simulate_event_slow(self, E, source_pos):
        num_photons = int(round(self.Y * E))
        total_photoelectrons = 0

        for _ in range(num_photons):
            z = np.random.uniform(-1.0, 1.0)
            phi = np.random.uniform(0.0, 2 * np.pi)
            sin_theta = np.sqrt(1.0 - z**2)
            dx = sin_theta * np.cos(phi)
            dy = sin_theta * np.sin(phi)
            dz = z

            b = 2.0 * (dx*source_pos[0] + dy*source_pos[1] + dz*source_pos[2])
            c = (source_pos[0]**2 + source_pos[1]**2 + source_pos[2]**2) - self.R**2
            t = (-b + np.sqrt(b**2 - 4.0*c)) / 2.0

            survival_prob = np.exp(-t / self.att_len)
            if np.random.uniform(0.0, 1.0) > survival_prob:
                continue

            hx = source_pos[0] + dx * t
            hy = source_pos[1] + dy * t
            hz = source_pos[2] + dz * t

            min_dist = float('inf')
            closest_pmt_idx = -1
            for idx, pmt in enumerate(self.pmts):
                dist = np.sqrt((hx-pmt[0])**2 + (hy-pmt[1])**2 + (hz-pmt[2])**2)
                if dist < min_dist:
                    min_dist = dist
                    closest_pmt_idx = idx

            if min_dist <= self.r:
                total_photoelectrons += np.random.poisson(lam=self.a)

        return total_photoelectrons

# 2. Векторная реализация
class FastDetectorAndSource:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5, att_len=8.0, Y=10000):
        self.R, self.N, self.r, self.a, self.att_len, self.Y = R, N, r, a, att_len, Y
        phi = np.pi * (5.0**0.5 - 1.0)
        points = []
        for i in range(N):
            y = 1 - (i / float(N - 1)) * 2
            r_at_y = np.sqrt(1 - y * y)
            theta = phi * i
            points.append([np.cos(theta)*r_at_y*R, y*R, np.sin(theta)*r_at_y*R])
        self.pmt_coordinates = np.array(points, dtype=np.float32)

    def simulate_event_fast(self, E, source_pos):
        num_photons = int(round(self.Y * E))

        source_pos = np.array(source_pos, dtype=np.float32)

        z = np.random.uniform(-1.0, 1.0, size=num_photons)
        phi = np.random.uniform(0.0, 2 * np.pi, size=num_photons)
        sin_theta = np.sqrt(1.0 - z**2)
        directions = np.stack([sin_theta * np.cos(phi), sin_theta * np.sin(phi), z], axis=1).astype(np.float32)

        b = 2.0 * np.dot(directions, source_pos)
        c = np.sum(source_pos**2) - self.R**2
        t = (-b + np.sqrt(b**2 - 4.0*c)) / 2.0

        survival_probability = np.exp(-t / self.att_len)
        survived_mask = np.random.uniform(0.0, 1.0, size=len(t)) < survival_probability
        directions = directions[survived_mask]
        t = t[survived_mask]

        if len(directions) == 0:
            return 0

        hit_points = source_pos + directions * t[:, np.newaxis]
        diff = hit_points[:, np.newaxis, :] - self.pmt_coordinates[np.newaxis, :, :]
        distances = np.linalg.norm(diff, axis=2)

        min_indices = np.argmin(distances, axis=1)
        min_distances = distances[np.arange(len(hit_points)), min_indices]

        valid_hits = min_indices[min_distances <= self.r]

        if len(valid_hits) == 0:
            return 0

        photoelectrons = np.random.poisson(lam=self.a, size=len(valid_hits))
        return int(np.sum(photoelectrons))

# Запуск и сравнение производительности

if __name__ == "__main__":
    E_test = 5.0
    pos_test = [0.3, -0.2, 0.8]
    n_iterations = 100

    slow_detector = SlowDetectorAndSource()
    fast_detector = FastDetectorAndSource()

    print(f"--- Производительность ({n_iterations} событий по {E_test} МэВ) ---")
    print(f"Каждое событие рождает порядка {int(E_test * 10000)} фотонов.\n")

    print("Тестирование медленного метода (циклы по отдельным фотонам)...")
    t0 = time.perf_counter()
    slow_charges = [slow_detector.simulate_event_slow(E_test, pos_test) for _ in range(n_iterations)]
    t_slow = time.perf_counter() - t0
    print(f"-> Медленный метод завершён за: {t_slow:.4f} сек.")

    print("\nТестирование векторизованного метода (массивы)...")
    t0 = time.perf_counter()
    fast_charges = [fast_detector.simulate_event_fast(E_test, pos_test) for _ in range(n_iterations)]
    t_fast = time.perf_counter() - t0
    print(f"-> Быстрый метод завершён за: {t_fast:.4f} сек.")

    speedup = t_slow / t_fast
    print("\n================ АНАЛИЗ ПРОИЗВОДИТЕЛЬНОСТИ ================")
    print(f"Время выполнения с циклами (List):     {t_slow:.4f} сек.")
    print(f"Время выполнения с NumPy (Array):      {t_fast:.4f} сек.")
    print(f"**Векторизованный NumPy работает быстрее в {speedup:.1f} раз**")
    print("===========================================================")


--- Производительность (100 событий по 5.0 МэВ) ---
Каждое событие рождает порядка 50000 фотонов.

Тестирование медленного метода (циклы по отдельным фотонам)...
-> Медленный метод завершён за: 331.3172 сек.

Тестирование векторизованного метода (массивы)...
-> Быстрый метод завершён за: 7.4003 сек.

================ АНАЛИЗ ПРОИЗВОДИТЕЛЬНОСТИ ================
Время выполнения с циклами (List):     331.3172 сек.
Время выполнения с NumPy (Array):      7.4003 сек.
**Векторизованный NumPy работает быстрее в 44.8 раз**
